# 00 Dataset triage

Every candidate is scored against the G1–G5 admission gate from the plan. A dataset is used
only if all five pass; a drop is recorded with the reason it failed, not just removed.

The point of doing this in code rather than in a paragraph is that the gate has to be
applied to every candidate *before* any of them is used.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## The registry

In [2]:
from swsn.registry import load_registry

registry = load_registry("../data/datasets.yaml")
pd.DataFrame([
    {
        "dataset": d.id,
        "class": d.cls.value,
        "domain": d.domain,
        "version": d.version,
        "licence": d.licence,
        "status": d.status.value,
        "files checksummed": len(d.sha256),
    }
    for d in registry.datasets.values()
]).sort_values(["class", "status", "dataset"])

,dataset,class,domain,version,licence,status,files checksummed
1,instacart,c_doc,grocery,1,CC0: Public Domain,admitted,6
0,olist,c_doc,e-commerce,2,CC BY-NC-SA 4.0,admitted,9
2,movielens,c_doc,movies,NaN,NaN,dropped,0
5,airbnb_nyc,c_nat,housing,3,CC0: Public Domain,admitted,1
4,cardekho_vehicles,c_nat,used-cars,4,"Database: Open Database, Contents: Database Co...",admitted,4
3,craigslist_cars,c_nat,used-cars,10,CC0: Public Domain,admitted,1
8,diabetes,c_nat,health,1,Data files © Original Authors,admitted,1
7,heart_failure,c_nat,health,1,"Database: Open Database, Contents: © Original ...",admitted,1
6,melbourne_housing,c_nat,housing,27,CC BY-NC-SA 4.0,admitted,2


## Why MovieLens was dropped

The slug `grouplens/movielens-25m-dataset` returns **HTTP 403** from the public Kaggle
metadata endpoint, so it is either not public or no longer exists under that name. It was
optional in the plan; Olist and Instacart already give the contrast the report needs, and
the days saved go to the C-nat annotation round.

## Licences are in the API, not on the page

The dataset page renders its metadata sidebar client-side, so fetching the HTML returns
only the title. `https://www.kaggle.com/api/v1/datasets/view/<owner>/<slug>` returns JSON
with `licenseName`, anonymously. Every licence above was read from there.

## Gate results for the admitted datasets

In [3]:
import json, pathlib
path = pathlib.Path("../evaluation/triage.json")
if path.exists():
    payload = json.loads(path.read_text())
    display(pd.DataFrame([r["summary"] for r in payload]))
    for r in payload:
        print(f"--- {r['summary']['dataset']}")
        for g in r["gates"]:
            print(f"  {g['gate']} {'pass' if g['passed'] else 'FAIL'}  {g['evidence']}")
else:
    print("run `python -m swsn.triage` first, or `make bench`")

,dataset,admitted,failed_gates,dimensions,n_correspondences
0,olist,True,[],"[D3, D5, D7]",14
1,instacart,True,[],"[D3, D6, D7]",8


--- olist
  G1 pass  public Dataset, version 2, 9 files checksummed, licence: CC BY-NC-SA 4.0
  G2 pass  not one of Valentine's benchmark sources or their tables
  G3 pass  14 correspondences transcribed (documented relationship required for c_doc)
  G4 pass  3 of D1-D7 present without synthetic corruption: D3, D5, D7 (need 3)
  G5 pass  9 columns, largest table 50,000 rows; sampleable to 5k rows on a shared key
--- instacart
  G1 pass  public Dataset, version 1, 6 files checksummed, licence: CC0: Public Domain
  G2 pass  not one of Valentine's benchmark sources or their tables
  G3 pass  8 correspondences transcribed (documented relationship required for c_doc)
  G4 pass  3 of D1-D7 present without synthetic corruption: D3, D6, D7 (need 3)
  G5 pass  7 columns, largest table 50,000 rows; sampleable to 5k rows on a shared key


## What the gate actually excluded

Nothing was excluded for being too messy — the two C-doc datasets pass the messiness gate
at exactly **3 of 7 dimensions**, which is the minimum. D1 (header noise), D2 (type
instability) and D4 (format heterogeneity) are absent from both at L0. That is the finding
that motivates the corruption ladder: the dimensions a matcher is most sensitive to are
precisely the ones real Kaggle data does not have.